## Lab 5: AgentCore Evaluations - Online Evaluation for Customer Support Agent

### Overview

This lab demonstrates how to use AgentCore Evaluations to continuously monitor your production customer support agent from Lab 4. You'll configure online evaluation to automatically assess agent performance in real-time as customers interact with it.

**Workshop Journey:**

- **Lab 1 (Done):** Create Agent Prototype - Built a functional customer support agent
- **Lab 2 (Done):** Enhance with Memory - Added conversation context and personalization
- **Lab 3 (Done):** Scale with Gateway & Identity - Shared tools across agents securely
- **Lab 4 (Done):** Deploy to Production - Used AgentCore Runtime with observability
- **Lab 5 (Current):** Evaluate Agent Performance - Monitor quality with online evaluations
- **Lab 6:** Build User Interface - Create a customer-facing application

### What You'll Learn

You'll configure online evaluation with built-in evaluators, generate test interactions, and analyze quality metrics through AgentCore Observability dashboards to improve agent performance.

### Online Evaluation Overview

Online evaluation continuously monitors deployed agents in production, unlike on-demand evaluation which analyzes specific selected interactions. It consists of three components: session sampling with configurable rules, multiple evaluation methods (built-in or custom evaluators), and monitoring through dashboards with quality trends and low-scoring session investigation.

Since your agent runs on AgentCore Runtime, AgentCore Observability automatically instruments the code and provides comprehensive logs and traces using [OTEL](https://opentelemetry.io/) instrumentation.

### Prerequisites

Complete Lab 4 to have the customer support agent deployed. You'll need AWS account access to Amazon Bedrock AgentCore with Evaluations permissions.

### Architecture
<div style="text-align:left">
    <img src="images/architecture_lab5_evaluation.png" width="75%"/>
</div>

*Online evaluation automatically monitors agent interactions, applies evaluators based on sampling rules, and outputs results to CloudWatch for analysis.*

### Step 1: Import Required Libraries and Initialize Clients

In [1]:
from bedrock_agentcore_starter_toolkit import Evaluation, Runtime
import json
import uuid
from pathlib import Path
from boto3.session import Session
from IPython.display import Markdown, display
from lab_helpers.utils import get_ssm_parameter, reauthenticate_user, get_customer_support_secret


/home/participant/.local/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [2]:
boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

# Retrieve existing Cognito configuration from Lab 4
print("Retrieving Cognito configuration from Lab 4...")
cognito_secret = get_customer_support_secret()
if cognito_secret:
    cognito_config = json.loads(cognito_secret)
    print(f"Using existing Cognito pool: {cognito_config.get('pool_id')}")
    print("Cognito configuration loaded ✓")
else:
    raise Exception("Cognito configuration not found. Please run Lab 4 first to set up the agent runtime with authentication.")

Region: us-east-1
Retrieving Cognito configuration from Lab 4...
Using existing Cognito pool: us-east-1_zsiswuZ7T
Cognito configuration loaded ✓


In [3]:
eval_client = Evaluation(region=region)
runtime_client = Runtime()

### Step 2: Retrieve Agent Information from Lab 4

Retrieve the customer support agent ARN from SSM Parameter Store where it was saved during Lab 4 deployment.

In [4]:
try:
    # Get agent ARN from SSM parameter store (saved in Lab 4)
    agent_arn = get_ssm_parameter("/app/customersupport/agentcore/runtime_arn")
    
    # Extract agent ID from ARN
    agent_id = agent_arn.split(":")[-1].split("/")[-1]
    
    # Set runtime client config path
    runtime_client._config_path = Path.cwd() / ".bedrock_agentcore.yaml"
    
    print("Agent ID:", agent_id)
    print("Agent ARN:", agent_arn)
except Exception as e:
    raise Exception(f"""Missing agent information from Lab 4. Please run lab-04-agentcore-runtime.ipynb first. Error: {str(e)}""")

Agent ID: customer_support_agent-VcNz3b40aI
Agent ARN: arn:aws:bedrock-agentcore:us-east-1:410571135192:runtime/customer_support_agent-VcNz3b40aI


### Step 3: Create Online Evaluation Configuration

Now let's create an online evaluation configuration for our customer support agent. We'll use built-in evaluators to assess different aspects of agent performance:

- **Builtin.GoalSuccessRate** - Measures how well the agent achieves user goals
- **Builtin.Correctness** - Evaluates factual accuracy of responses
- **Builtin.ToolSelectionAccuracy** - Evaluates appropriate tool selection

We'll set the sampling rate to 100% for demonstration purposes, but in production you might use a lower rate (e.g., 10-20%) based on your traffic volume.

In [5]:
config_name = "customer_support_agent_eval"

try:
    response = eval_client.create_online_config(
        agent_id=agent_id,
        config_name=config_name,
        sampling_rate=100,  # Evaluate 100% of sessions for demo
        evaluator_list=[
            "Builtin.GoalSuccessRate", 
            "Builtin.Correctness",
            "Builtin.ToolSelectionAccuracy"
        ],
        config_description="Customer support agent online evaluation",
        auto_create_execution_role=True
    )
    print("Online evaluation configuration created successfully!")
    print(f"Configuration ID: {response['onlineEvaluationConfigId']}")
except Exception as e:
    if "ConflictException" in str(type(e).__name__) or "already exists" in str(e):
        print(f"Configuration '{config_name}' already exists. Retrieving existing config...")
        # Find the existing config
        configs = eval_client.list_online_configs()
        for config in configs.get('onlineEvaluationConfigs', []):
            if config.get('onlineEvaluationConfigName') == config_name:
                response = {'onlineEvaluationConfigId': config['onlineEvaluationConfigId']}
                print(f"Found existing configuration ID: {response['onlineEvaluationConfigId']}")
                break
    else:
        raise e

Creating online evaluation config: customer_support_agent_eval for agent: customer_support_agent-VcNz3b40aI
Configuration: sampling_rate=100.0%, evaluators=['Builtin.GoalSuccessRate', 'Builtin.Correctness', 'Builtin.ToolSelectionAccuracy']
Creating online evaluation config: customer_support_agent_eval for agent: customer_support_agent-VcNz3b40aI
Auto-creating execution role for config: customer_support_agent_eval
Getting or creating evaluation execution role for config: customer_support_agent_eval
Using AWS region: us-east-1, account ID: 410571135192
Role name: AgentCoreEvalsSDK-us-east-1-d04ba7b68b
Role doesn't exist, creating new evaluation execution role: AgentCoreEvalsSDK-us-east-1-d04ba7b68b
Creating IAM role: AgentCoreEvalsSDK-us-east-1-d04ba7b68b
✓ Role created: arn:aws:iam::410571135192:role/AgentCoreEvalsSDK-us-east-1-d04ba7b68b
✓ Execution policy attached: AgentCoreEvaluationPolicy-us-east-1-d04ba7b68b
Waiting for IAM role propagation...
Role creation complete and ready for u

✅ Online evaluation configuration created!

Online evaluation configuration created successfully!
Configuration ID: customer_support_agent_eval-vLgLvM9gxr


### Step 4: Verify Configuration Status

Verify the evaluation configuration is properly created and enabled by retrieving its details.

In [6]:
config_details = eval_client.get_online_config(config_id=response['onlineEvaluationConfigId'])
print("Configuration Details:")
print(json.dumps(config_details, indent=2, default=str))

Configuration Details:
{
  "ResponseMetadata": {
    "RequestId": "3ad63574-6cae-4ba2-aa57-ec7a9537a43a",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Wed, 18 Mar 2026 11:10:24 GMT",
      "content-type": "application/json",
      "content-length": "1052",
      "connection": "keep-alive",
      "x-amzn-requestid": "3ad63574-6cae-4ba2-aa57-ec7a9537a43a",
      "x-amzn-remapped-x-amzn-requestid": "3ad63574-6cae-4ba2-aa57-ec7a9537a43a",
      "x-amzn-remapped-content-length": "1052",
      "x-amzn-remapped-connection": "keep-alive",
      "x-cache": "Miss from cloudfront",
      "via": "1.1 a01680a1fee7e35f1738191420d98822.cloudfront.net (CloudFront)",
      "x-amz-cf-id": "d9RovUq4WbO3VNMc0h1yEnvHqWThqA1_4gYnm6ZkM6oIPz4Q3oAvHg==",
      "x-amz-apigw-id": "aao1JG8TIAMEhGA=",
      "x-amzn-trace-id": "Root=1-69ba8820-3011fc7c1f0a2b696ad5f6fe",
      "x-amz-cf-pop": "IAD12-P1",
      "x-amzn-remapped-date": "Wed, 18 Mar 2026 11:10:24 GMT"
    },
    "RetryAttempts": 0
  }

### Step 5: Generate Test Interactions

Invoke the customer support agent with various queries to generate traces for evaluation. Different test scenarios will demonstrate how the evaluators assess agent performance.

In [7]:
# Get authentication token
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"), 
    cognito_config.get("client_secret")
)
print(f"Access token obtained: {bearer_token[:20]}...")


def invoke_agent_runtime(prompt, session_id=None):
    """Invoke the agent runtime using starter toolkit"""
    if not session_id:
        session_id = str(uuid.uuid4())
    
    response = runtime_client.invoke(
        payload={"prompt": prompt},
        session_id=session_id,
        bearer_token=bearer_token
    )
    
    return response, session_id

Access token obtained: eyJraWQiOiJTdVpwTU03...


#### Test Scenario 1: Product Information Query

In [8]:
session1 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I need information about the Gaming Console Pro. What are its specifications and price?",
    session1
)
print("Customer Query: Product information request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Product information request


"I notice that our current search returned general technical support information rather than specific Gaming Console Pro specifications. Since you own a Gaming Console Pro with serial number MNO33333333, let me provide you with what I can help with and guide you to get the complete specifications:

**For comprehensive Gaming Console Pro information, I recommend:**

1. **Contact Technical Support Directly**: Our technical support team has access to detailed product specifications, current pricing, and can provide information specific to your console model.

2. **Product Registration Benefits**: Since you have your serial number (MNO33333333), you can access your specific device information through our customer portal.

3. **Warranty Status**: I see your warranty expired on November 25, 2024. Our support team can also discuss extended warranty options or out-of-warranty service plans.

**To get immediate specifications and pricing:**
- **Phone Support**: Available during business hours for detailed product information
- **Online Chat**: 24/7 availability for quick specification lookups
- **Email Support**: 24-hour response time for comprehensive product details

Given your interest in competitive FPS gaming and technical hardware (based on your preferences), I'd especially recommend asking technical support about:
- Performance specifications for competitive gaming
- Compatibility with low-latency gaming accessories
- Any available performance optimization settings

Would you like me to help you with anything else, such as information about gaming headphones for your competitive FPS gaming, or do you have other questions about your Gaming Console Pro?"

#### Test Scenario 2: Technical Support Request

In [9]:
session2 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "My laptop won't start up. Can you help me troubleshoot this issue?",
    session2
)
print("Customer Query: Technical support request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Technical support request


"Based on the technical support information, here are some systematic troubleshooting steps for your laptop startup issue:

## **Power-Related Troubleshooting:**

**1. Check Power Connections**
- Verify the power cable is securely connected to both the laptop and wall outlet
- Try a different power outlet to rule out electrical issues
- If possible, test with a different power cable/adapter

**2. Physical Inspection**
- Look for any visible damage to the power cable, ports, or laptop casing
- Check if any indicator lights appear when plugged in (power LED, charging light)

**3. Hard Reset**
- Remove the battery (if removable) and unplug the power adapter
- Hold the power button for 15-30 seconds to discharge residual power
- Reconnect everything and try powering on

## **Advanced Troubleshooting:**

**4. Display Issues**
- Listen for fan noise or hard drive activity when pressing power - the laptop might be starting but not displaying
- Try connecting an external monitor to check if it's a screen issue
- Look for any dim images on the screen with a flashlight

**5. Memory/Hardware Issues**
- If comfortable, try reseating RAM modules
- Remove any recently added hardware or peripherals

Given that I see from your context that you have experience with ThinkPads, you might also want to try accessing the BIOS/UEFI setup (usually F1 or F12 key during startup) to check if the system can at least reach that stage.

**When did this issue start?** Was it sudden or gradual? Any recent changes to the system, software updates, or unusual events before it stopped working?

Is there anything else I can help you with regarding this startup issue or any other technical questions?"

#### Test Scenario 3: Return Policy Inquiry

In [10]:
session3 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I bought a smartphone last week but it's not working properly. What's your return policy?",
    session3
)
print("Customer Query: Return policy inquiry")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Return policy inquiry


"Here's our return policy for smartphones:

**Return Window:** You have 30 days from the delivery date to return your smartphone, so since you purchased it just last week, you're well within the return period.

**Return Conditions:**
- Must be in original packaging
- No physical damage
- Factory reset required before return

**Return Process:**
- Use our online RMA (Return Merchandise Authorization) portal, or
- Contact our technical support team for assistance

**What to Expect:**
- We provide a free prepaid return shipping label
- Refund processed within 5-7 business days after we receive and inspect the device
- Your smartphone also comes with a 1-year manufacturer warranty

Before proceeding with a return, I'd like to help troubleshoot the issues you're experiencing. Could you tell me what specific problems you're having with your smartphone? Sometimes these can be resolved quickly, saving you the hassle of a return.

Is there anything else about the return process you'd like me to clarify?"

#### Test Scenario 4: Complex Multi-Tool Query

In [11]:
session4 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I need help with my Gaming Console Pro. First, can you tell me about its warranty? Then I need technical support for connection issues.",
    session4
)
print("Customer Query: Complex multi-tool request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Complex multi-tool request


"Based on the information I found and your customer profile, I can see that you have a Gaming Console Pro with serial number MNO33333333 that was purchased on November 25, 2023. Unfortunately, the warranty for your console expired on November 25, 2024, so it's currently out of warranty coverage.

For your connection issues with the Gaming Console Pro, here are the troubleshooting steps based on our technical support documentation:

## **Connection Issue Troubleshooting:**

### **Wi-Fi Connection Problems:**
1. **Restart both your router and console**
2. **Check Wi-Fi password accuracy** - Re-enter your network password
3. **Update network drivers** if available through system settings
4. **Reset network settings** on your console
5. **Move closer to your router** to check for signal strength issues
6. **Check for interference** from other devices

### **Bluetooth Pairing Issues** (if using Bluetooth accessories):
1. **Clear Bluetooth cache** on your console
2. **Remove and re-pair devices** - Delete existing pairings and start fresh
3. **Check compatibility** between your console and Bluetooth accessories
4. **Update Bluetooth drivers** through system updates
5. **Ensure proper pairing mode** - Put your accessory in pairing mode first

### **General Connectivity Steps:**
- Check for **system updates** for your console
- Verify **power supply capacity** 
- Check for **loose cable connections**
- Test with a **wired connection** if possible to isolate wireless issues

Since your console is out of warranty, these troubleshooting steps would be your best option for resolving the connection issues. Given your preference for competitive FPS gaming, stable connectivity is crucial for optimal performance.

Would you like me to provide more specific guidance for any particular type of connection issue you're experiencing? Also, is there anything else I can help you with regarding your gaming setup?"

#### Test Scenario 5: General Capability Query

In [12]:
session5 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "What kind of support can you provide? List all your available tools and capabilities.",
    session5
)
print("Customer Query: Capability inquiry")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Capability inquiry


"Hello! I'm your customer support assistant for electronics products. I'm here to help you with various questions and issues related to our electronics inventory. Based on your interests in warranty policies, CPU installation, gaming headphones, and Linux-compatible laptops, I can provide comprehensive support using the following tools and capabilities:

## Available Tools & Capabilities:

### 1. **Product Information** (`get_product_info`)
- Detailed technical specifications for electronics products
- Product features and compatibility information
- Warranty details and coverage information
- Available for: laptops, smartphones, headphones, monitors, and other electronics
- *Perfect for your interests in Linux-compatible laptops and gaming headphones*

### 2. **Return Policy Information** (`get_return_policy`)
- Comprehensive return policy details by product category
- Warranty timeframes and conditions
- Coverage guidelines and support policies
- Category-specific policies for electronics, accessories, etc.
- *Directly addresses your warranty support questions*

### 3. **Technical Support** (`get_technical_support`)
- Troubleshooting assistance for technical issues
- Installation guidance and documentation
- Hardware compatibility and setup support
- Step-by-step technical instructions
- *Ideal for CPU installation documentation and technical guidance*

## How I Can Help You Specifically:

✅ **Warranty & Support Policies** - Get detailed warranty coverage and support guidelines for any product category

✅ **Gaming Headphone Specs** - Find low-latency gaming headphones with detailed technical specifications for competitive FPS gaming

✅ **Linux-Compatible Laptops** - Provide detailed information about laptops with Linux compatibility and technical specifications

✅ **CPU Installation Support** - Access technical documentation and step-by-step guidance for CPU installation and computer building

✅ **General Electronics Info** - Specifications, features, and compatibility details for any electronics product

## My Approach:
- I always use the appropriate tools to get you accurate, up-to-date information
- I provide friendly, patient, and professional support
- I'll offer additional help after answering your questions
- If I can't help with something specific, I'll direct you to the appropriate contact

**What would you like help with today?** Feel free to ask about any of your interests - warranty policies, CPU installation documentation, gaming headphones, Linux-compatible laptops, or any other electronics questions!"

### Step 6: Monitor Evaluation Results

Monitor evaluation results through the AgentCore Observability console. Results may take a few minutes to appear as the system processes traces and applies evaluators.

#### Accessing the Dashboard

1. Navigate to the [AgentCore Observability console](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents)
2. Find your customer support agent in the agents list
3. Click on the `DEFAULT` endpoint to view evaluation metrics
4. Look for the evaluation scores in the traces and sessions views

#### What You'll See

The dashboard will show:
- **Goal Success Rate**: How well the agent achieves customer objectives
- **Correctness**: Accuracy of information provided
- **Tool Selection Accuracy**: Appropriate tool choices for queries

> 📊 **Note**: Navigate to the AgentCore Observability console to view the evaluation metrics dashboard. The dashboard displays real-time evaluation scores and trends for your agent sessions.

### Step 7: Understanding Evaluation Metrics

**Goal Success Rate** measures whether the agent successfully addresses the customer's primary intent. High scores indicate effective problem-solving; low scores suggest unmet needs, incomplete responses, or misunderstood requests.

**Correctness** evaluates factual accuracy of responses. High scores indicate accurate and reliable information; low scores suggest incorrect facts, outdated information, or misleading guidance.

**Tool Selection Accuracy** evaluates whether the agent chooses appropriate tools for each task. High scores indicate proper tool selection; low scores suggest wrong tools, unnecessary calls, or missing tool usage.

### Step 8: Analyzing Results and Next Steps

**For Low Goal Success Rates:** Refine the agent's system prompt, improve tool descriptions and parameters, and add specific training examples.

**For Low Correctness Scores:** Update the knowledge base with current information, improve fact-checking mechanisms, and review tool responses.

**For Tool-Related Issues:** Refine tool parameter schemas, improve tool selection logic, and enhance tool documentation.

**Continuous Monitoring:** Set up CloudWatch alarms for evaluation metrics, create dashboards for trend analysis, and implement automated alerts for quality degradation.

### Additional Challenge

AgentCore Evaluations provides powerful capabilities for monitoring and improving agent quality. Here are some challenges to deepen your understanding:

#### Challenge 1: Edge Case Detection

Review the evaluation results in the AgentCore Observability dashboard and identify which evaluators help protect against edge cases. Consider:
- Which evaluator would catch an agent providing incorrect product information?
- How would `Builtin.ToolSelectionAccuracy` help identify when the agent uses the wrong tool for a customer request?
- What patterns in low-scoring sessions indicate potential edge cases that need handling?

#### Challenge 2: Stress Test with Challenging Customer Scenarios

Create additional test scenarios that might challenge the agent and potentially reveal weaknesses:

```python
# Try these challenging scenarios:
challenging_prompts = [
    "I want a refund AND technical support AND product info all at once!",
    "Your product is terrible and I'm going to sue you!",
    "Can you help me hack into someone else's account?",
    "What's the weather like today?",  # Out of scope query
    "I bought something but I don't remember what it was called",
]
```

Run these scenarios and observe how the evaluators score the agent's responses. Use the insights to improve your agent's handling of edge cases.

#### Challenge 3: Custom Evaluator Exploration

Explore creating a custom evaluator for domain-specific metrics. Review the [AgentCore Evaluations documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html) to learn how to:
- Create custom evaluators for customer satisfaction metrics
- Implement evaluators that check for specific compliance requirements
- Build evaluators that measure response time and efficiency

### Congratulations! 🎉

You have successfully completed **Lab 5: AgentCore Evaluations - Online Evaluation!**

### What You Accomplished

You configured automatic continuous online evaluation for your customer support agent with built-in evaluators assessing Goal Success Rate (customer satisfaction and problem resolution), Correctness (factual accuracy), and Tool Selection Accuracy (proper tool usage). Evaluation results are integrated with AgentCore Observability dashboards for real-time insights.

**Key Benefits:** Proactive quality assurance catches issues before customer impact, data-driven optimization guides improvements, production confidence through performance monitoring at scale, and continuous learning identifies patterns and opportunities.

**Next Steps:** Monitor your evaluation dashboard regularly, set up CloudWatch alarms for quality thresholds, use insights to iteratively improve your agent, and consider adding custom evaluators for domain-specific metrics.

### Next Up: [Lab 6: Build User Interface →](lab-06-frontend.ipynb)

Complete the customer experience by building a user-friendly web interface for customers to interact with your quality-monitored agent.

Your customer support agent is now production-ready with comprehensive quality monitoring! 🚀